# 🌾 Crop Yield Prediction — Capstone Walkthrough

This notebook documents the end-to-end crop-yield prediction workflow used by the project. It is intentionally aligned with the repository's current training pipeline and reported evaluation files.

**Objective:** predict `yield_tonnes_ha` from agricultural, environmental, and farm-management variables.


## 1. Project Workflow

1. Load and validate the dataset  
2. Inspect distributions and missing values  
3. Prepare numeric and categorical features  
4. Compare Ridge, Random Forest, and Gradient Boosting  
5. Evaluate on a holdout test set and with 5-fold cross-validation  
6. Interpret the selected model with permutation importance  
7. Use the saved model in the Streamlit application


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA_PATH = ROOT / 'data' / 'crop_yield_prediction.csv'
METRICS_PATH = ROOT / 'reports' / 'evaluation.json'
COMPARISON_PATH = ROOT / 'reports' / 'model_comparison.csv'
IMPORTANCE_PATH = ROOT / 'reports' / 'feature_importance.csv'

print('Repository root:', ROOT)


In [ ]:
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
display(df.head())


## 2. Data Quality Check


In [ ]:
quality = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing': df.isna().sum(),
    'unique_values': df.nunique(),
})
display(quality)


In [ ]:
target = 'yield_tonnes_ha'
display(df[target].describe().to_frame('yield_tonnes_ha'))

plt.figure(figsize=(8, 4))
plt.hist(df[target], bins=30)
plt.xlabel('Yield (tonnes/hectare)')
plt.ylabel('Frequency')
plt.title('Crop Yield Distribution')
plt.tight_layout()
plt.show()


## 3. Crop-Level Exploration


In [ ]:
crop_summary = (df.groupby('crop')[target]
                .agg(['count', 'mean', 'median', 'std'])
                .sort_values('mean', ascending=False))
display(crop_summary)

plt.figure(figsize=(10, 5))
crop_summary['mean'].plot(kind='bar')
plt.ylabel('Mean yield (tonnes/hectare)')
plt.title('Average Yield by Crop')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 4. Model Comparison


In [ ]:
comparison = pd.read_csv(COMPARISON_PATH)
display(comparison)

comparison_plot = comparison.set_index('model')[['mae', 'rmse']]
comparison_plot.plot(kind='bar', figsize=(9, 5))
plt.ylabel('Error (tonnes/hectare)')
plt.title('Holdout Model Comparison')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 5. Cross-Validation and Final Evaluation

The repository uses a fixed 80/20 holdout split for the reported test metrics and five-fold cross-validation to provide a more robust estimate of model stability.


In [ ]:
metrics = json.loads(METRICS_PATH.read_text(encoding='utf-8'))
print('Selected model:', metrics['selected_model'])
print('Test metrics:')
display(pd.DataFrame([metrics['test_metrics']]))
print(f"5-fold CV RMSE: {metrics['five_fold_cv_rmse_mean']:.3f} ± {metrics['five_fold_cv_rmse_std']:.3f} tonnes/hectare")


## 6. Feature Importance

Permutation importance estimates how much the model's MAE increases when each feature is shuffled. It indicates predictive reliance, not causality.


In [ ]:
importance = pd.read_csv(IMPORTANCE_PATH).sort_values('mae_increase', ascending=True)
display(importance)

plt.figure(figsize=(9, 6))
plt.barh(importance['feature'], importance['mae_increase'])
plt.xlabel('Increase in MAE after permutation')
plt.title('Permutation Feature Importance')
plt.tight_layout()
plt.show()


## 7. Findings

- Gradient Boosting is the selected model in the current pipeline.
- The holdout evaluation reports strong predictive performance on the supplied dataset.
- Five-fold cross-validation is included to assess stability beyond a single split.
- Feature importance highlights which inputs the fitted model relies on most.
- The high R² should be interpreted cautiously until the model is validated on an independent external dataset.


## 8. Limitations and Next Steps

The dataset is limited to the supplied observations and has not been independently validated against field measurements. Future work should investigate data provenance, leakage, geographic/seasonal generalization, repeated cross-validation, hyperparameter tuning, and independent external validation.


## 9. Running the Project

From the repository root:

```bash
python -m pip install -r requirements.txt
python -m pytest -q
python src/train.py --data data/crop_yield_prediction.csv
python -m streamlit run app.py
```
